# Itérateurs et générateurs — traiter des données sans tout charger en mémoire

Le notebook « performance » a montré *qu'un* générateur économise la mémoire.
Ici on ouvre le capot : **comment** la boucle `for` fonctionne vraiment, comment écrire
ses propres générateurs, et comment les **chaîner** pour traiter des gros volumes
élément par élément.

### Vocabulaire (simple)
- **Itérable** : tout ce qu'on peut parcourir avec un `for` (liste, string, fichier, range...).
  Analogie : un **livre** — on peut le lire autant de fois qu'on veut.
- **Itérateur** : l'objet qui **fait** le parcours, en retenant où il en est.
  Analogie : un **marque-page** — il avance page par page, et ne revient jamais en arrière.
- **Générateur** : la façon la plus simple de fabriquer un itérateur, avec le mot-clé `yield`.

---
## 1. Ce que fait vraiment une boucle `for`

Quand tu écris `for x in ma_liste:`, Python fait en réalité :

1. `iter(ma_liste)` → demande un **itérateur** (le marque-page)
2. `next(iterateur)` → demande l'élément suivant, encore et encore
3. Quand il n'y a plus rien → l'itérateur lève `StopIteration`, le `for` s'arrête proprement

La boucle `for` n'est donc que du sucre autour de `iter()` + `next()` :

In [ ]:
fruits = ["pomme", "poire", "kiwi"]

# Ce que le for fait sous le capot :
iterateur = iter(fruits)          # 1. obtenir le marque-page
print(next(iterateur))            # 2. pomme
print(next(iterateur))            #    poire
print(next(iterateur))            #    kiwi

try:
    next(iterateur)               # 3. plus rien →
except StopIteration:
    print("StopIteration : le for s'arrêterait ici")

# Un itérateur s'épuise : une fois au bout, il est fini pour de bon
print(f"\nRelire le même itérateur : {list(iterateur)}")   # [] — épuisé
print(f"Relire la liste (itérable) : {list(fruits)}")      # la liste, elle, est toujours là

---
## 2. Écrire un générateur avec `yield`

Une fonction qui contient `yield` devient une **fonction génératrice** : l'appeler ne l'exécute pas,
ça fabrique un générateur. Le code ne tourne qu'au fur et à mesure des `next()` :

| Mot-clé | Comportement |
|---|---|
| `return` | termine la fonction, tout est fini |
| `yield` | **met en pause** la fonction, livre une valeur, **reprendra exactement ici** au prochain `next()` |

C'est la pause/reprise qui fait l'économie de mémoire : **une seule valeur existe à la fois**.

In [ ]:
def compte_a_rebours(depart):
    print(f"  (démarrage à {depart})")
    n = depart
    while n > 0:
        yield n          # pause ici — reprendra à la ligne suivante
        n -= 1
    print("  (générateur terminé)")


g = compte_a_rebours(3)
print("Le générateur est créé mais RIEN n'a encore tourné.")
print(f"next → {next(g)}")   # exécute jusqu'au premier yield
print(f"next → {next(g)}")   # reprend après le yield, re-boucle
print(f"next → {next(g)}")

# Dans un for, c'est transparent :
print("\nAvec un for :")
for n in compte_a_rebours(3):
    print(n, end=" ")

### Un générateur peut même être infini

Impossible avec une liste (mémoire finie !) — trivial avec un générateur,
puisqu'il ne produit les valeurs qu'à la demande.

In [ ]:
import itertools

def nombres_pairs():
    n = 0
    while True:          # boucle infinie — aucun problème : rien n'est stocké
        yield n
        n += 2

# islice = « prendre une tranche » d'un itérateur (comme liste[0:5], mais sans liste)
cinq_premiers = itertools.islice(nombres_pairs(), 5)
print(list(cinq_premiers))   # [0, 2, 4, 6, 8]

---
## 3. Les pipelines — chaîner des générateurs

Le vrai pouvoir des générateurs : on peut les **brancher les uns sur les autres**,
comme une chaîne de montage. Chaque élément traverse **toute la chaîne un par un** —
à aucun moment l'ensemble des données n'est en mémoire.

Cas type en entreprise : traiter un fichier de logs de plusieurs Go ligne par ligne.

In [ ]:
# Simulation : un « fichier » de logs (en vrai, ce serait open(...) — qui est déjà un itérable !)
logs_bruts = [
    "2026-09-01 10:00:01 INFO  démarrage du service",
    "2026-09-01 10:00:05 ERROR connexion base de données refusée",
    "2026-09-01 10:00:06 INFO  nouvelle tentative",
    "2026-09-01 10:00:09 ERROR timeout après 3 secondes",
    "2026-09-01 10:00:15 INFO  connexion établie",
    "2026-09-01 10:01:22 ERROR division par zéro dans le module facturation",
]

# Étape 1 : nettoyer  (générateur : rien n'est calculé ici)
lignes = (ligne.strip() for ligne in logs_bruts)

# Étape 2 : filtrer les erreurs  (branché sur l'étape 1)
erreurs = (ligne for ligne in lignes if " ERROR " in ligne)

# Étape 3 : extraire le message  (branché sur l'étape 2)
messages = (ligne.split(" ERROR ")[1] for ligne in erreurs)

# Jusqu'ici : AUCUN travail effectué. Tout se déclenche à la consommation :
for msg in messages:
    print(f"⚠ {msg}")

# Chaque ligne a traversé les 3 étapes une par une.
# Avec un fichier de 10 Go : même code, même consommation mémoire (quelques Ko).

### `yield from` — déléguer à un autre générateur

`yield from source` équivaut à `for x in source: yield x` — utile pour composer des générateurs
ou parcourir des structures imbriquées.

In [ ]:
def aplatir(structure):
    """Parcourt une liste de listes (de listes...) et livre les éléments un à un."""
    for element in structure:
        if isinstance(element, list):
            yield from aplatir(element)   # délègue au sous-parcours
        else:
            yield element


imbrique = [1, [2, 3, [4, 5]], 6, [[7], 8]]
print(list(aplatir(imbrique)))   # [1, 2, 3, 4, 5, 6, 7, 8]

---
## 4. `itertools` — la boîte à outils des itérateurs

Le module `itertools` fournit des briques d'itération écrites en C (donc rapides)
qui travaillent **sans créer de listes intermédiaires**.

| Fonction | Rôle |
|---|---|
| `islice(it, n)` | prendre les n premiers (ou une tranche) |
| `chain(a, b)` | enchaîner plusieurs itérables comme un seul |
| `batched(it, n)` | découper en paquets de n (Python 3.12+) |
| `pairwise(it)` | livrer les éléments par paires glissantes (a,b), (b,c)... |
| `count(start)` | compteur infini |
| `groupby(it, cle)` | regrouper les éléments consécutifs par clé |

In [ ]:
from itertools import chain, islice, pairwise, batched

equipe_a = ["Alice", "Bob"]
equipe_b = ["Chloé", "David", "Emma"]

# chain : parcourir plusieurs collections comme une seule (sans les fusionner en mémoire)
print("chain    :", list(chain(equipe_a, equipe_b)))

# pairwise : comparer chaque élément au suivant (ex : détecter une variation)
temperatures = [18, 21, 19, 25, 24]
variations = [apres - avant for avant, apres in pairwise(temperatures)]
print("pairwise :", variations)

# batched : traiter par paquets (ex : insérer en base 2 par 2)
print("batched  :", [list(paquet) for paquet in batched(range(7), 2)])

# islice : ne prendre que le début d'un flux potentiellement énorme
print("islice   :", list(islice((x ** 2 for x in range(10 ** 9)), 5)))   # instantané !

---
## 5. Bonus : rendre SA classe parcourable avec `__iter__`

Pour qu'un objet maison fonctionne dans un `for`, il suffit de lui donner une méthode `__iter__`
qui retourne un itérateur — le plus simple étant... d'en faire un générateur.

In [ ]:
class Playlist:
    def __init__(self, nom):
        self.nom = nom
        self._titres = []

    def ajouter(self, titre):
        self._titres.append(titre)

    def __iter__(self):
        # Un générateur = un itérateur tout fait. On livre les titres numérotés.
        for numero, titre in enumerate(self._titres, start=1):
            yield f"{numero}. {titre}"


playlist = Playlist("Focus")
playlist.ajouter("Lo-fi beats")
playlist.ajouter("Piano calme")
playlist.ajouter("Pluie d'orage")

for ligne in playlist:            # possible car __iter__ existe
    print(ligne)

print(list(playlist))             # et tout ce qui consomme un itérable marche aussi

---
## Récapitulatif

- Le `for` appelle `iter()` puis `next()` jusqu'à `StopIteration` — rien de magique.
- **Itérable** = le livre (relisible) ; **itérateur** = le marque-page (à usage unique).
- `yield` met la fonction **en pause** : une seule valeur en mémoire à la fois.
- Les générateurs se **chaînent** en pipelines : idéal pour les gros fichiers et les flux.
- `yield from` délègue ; `itertools` fournit les briques rapides écrites en C.
- `__iter__` (souvent écrit comme un générateur) rend n'importe quelle classe parcourable.

> **En entretien** : « Un générateur produit ses valeurs à la demande au lieu de tout stocker :
> mémoire O(1) au lieu de O(n). En contrepartie il ne se parcourt qu'une seule fois
> et n'a pas d'accès par index. Pour un traitement de fichier volumineux, je chaîne
> des générateurs : chaque ligne traverse le pipeline une par une. »